# Lab 8A: Building Data Pipelines - SOLUTION

This notebook contains the solution for Lab 8A exercises.

## 💻 Exercise 1: Build Complete dlt Pipeline - SOLUTION

### Solution

In [ ]:
# Solution framework for dlt pipeline
# Requires: pip install dlt[duckdb]

import duckdb
import pandas as pd

# Simulated dlt pipeline using DuckDB directly
def build_dlt_pipeline_simulation():
    """Build a complete dlt-like pipeline"""
    con = duckdb.connect('data/dlt_simulation.db')
    
    # Step 1: Define data source
    def fetch_api_data():
        # Simulate API data
        data = []
        for i in range(1000):
            data.append({
                "id": i + 1,
                "customer_id": (i % 100) + 1,
                "product_id": (i % 50) + 1,
                "amount": round(100 + (i % 400), 2),
                "status": ['pending', 'processing', 'shipped', 'delivered'][i % 4]
            })
        return pd.DataFrame(data)
    
    # Step 2: Create pipeline (simulate dlt pipeline)
    source_data = fetch_api_data()
    con.register('source_data', source_data)
    
    # Step 3: Load data into DuckDB
    con.execute("CREATE TABLE raw_source AS SELECT * FROM source_data")
    
    # Step 4: Explore loaded data
    count = con.execute("SELECT COUNT(*) FROM raw_source").fetchone()[0]
    print(f"✓ Pipeline loaded {count} records")
    
    # Step 5: Examine pipeline metadata
    con.execute("CREATE TABLE _dlt_pipeline_state AS
                 SELECT 'pipeline_simulation' as pipeline_name,
                        COUNT(*) as row_count,
                        NOW() as load_time
                 FROM raw_source")
    
    metadata = con.execute("SELECT * FROM _dlt_pipeline_state").fetchdf()
    print("✓ Pipeline metadata:")
    print(metadata)
    
    con.close()
    return True

# Execute the solution
print("dlt Pipeline Solution:")
print("="*50)
build_dlt_pipeline_simulation()

## 💻 Exercise 2: Create dbt Transformation Project - SOLUTION

### Solution

In [ ]:
# Solution for dbt-like transformations using DuckDB

def create_dbt_like_project():
    """Create a dbt-like transformation project"""
    con = duckdb.connect('data/dbt_simulation.db')
    
    # Step 1: Define sources (simulate dbt sources.yml)
    con.execute("""
        CREATE TABLE source_raw AS
        SELECT * FROM 'data/sample/orders.parquet'
    """)
    print("✓ Defined source: source_raw")
    
    # Step 2: Create staging models (simulate dbt staging)
    con.execute("""
        CREATE TABLE stg_orders AS
        SELECT 
            order_id,
            customer_id,
            product_id,
            order_date,
            delivery_date,
            quantity,
            unit_price,
            total_amount,
            status,
            payment_method
        FROM source_raw
    """)
    print("✓ Created staging model: stg_orders")
    
    # Step 3: Create intermediate models (simulate dbt intermediate)
    con.execute("""
        CREATE TABLE int_order_stats AS
        WITH orders AS (
            SELECT * FROM stg_orders
        ),
        daily_stats AS (
            SELECT 
                order_date,
                COUNT(*) as order_count,
                SUM(total_amount) as daily_revenue,
                AVG(total_amount) as avg_order_value
            FROM orders
            GROUP BY order_date
        )
        SELECT * FROM daily_stats
    """)
    print("✓ Created intermediate model: int_order_stats")
    
    # Step 4: Create final models (simulate dbt final)
    con.execute("""
        CREATE TABLE fct_customer_orders AS
        SELECT 
            customer_id,
            COUNT(*) as total_orders,
            SUM(total_amount) as total_spent,
            AVG(total_amount) as avg_order_value,
            MIN(order_date) as first_order,
            MAX(order_date) as last_order
        FROM stg_orders
        GROUP BY customer_id
    """)
    print("✓ Created final model: fct_customer_orders")
    
    # Step 5: Add tests (simulate dbt tests)
    test_results = []
    
    # Test: Not null
    null_check = con.execute("SELECT COUNT(*) FROM stg_orders WHERE customer_id IS NULL").fetchone()[0]
    test_results.append(('customer_id not null', null_check == 0))
    
    # Test: Unique
    unique_check = con.execute("SELECT COUNT(DISTINCT order_id) FROM stg_orders").fetchone()[0]
    total_check = con.execute("SELECT COUNT(*) FROM stg_orders").fetchone()[0]
    test_results.append(('order_id unique', unique_check == total_check))
    
    print("✓ Test results:")
    for test_name, result in test_results:
        status = "PASS" if result else "FAIL"
        print(f"  {test_name}: {status}")
    
    con.close()
    return True

# Execute the solution
print("dbt Transformation Project Solution:")
print("="*50)
create_dbt_like_project()

## 💻 Exercise 3: Build Dagster Pipeline - SOLUTION

### Solution

In [ ]:
# Solution for Dagster-like pipeline

def build_dagster_like_pipeline():
    """Build a complete Dagster-like pipeline"""
    con = duckdb.connect('data/dagster_simulation.db')
    
    # Step 1: Define extraction assets
    def asset_extract_customers():
        """Extract customer data"""
        df = con.execute("SELECT * FROM 'data/sample/customers.parquet'").df()
        print(f"✓ Extracted {len(df)} customers")
        return df
    
    # Step 2: Define transformation assets
    def asset_transform_sales(customers_df):
        """Transform sales data with customer enrichment"""
        con.register('customers', customers_df)
        
        result = con.execute("""
            SELECT 
                c.customer_id,
                c.segment,
                c.loyalty_points,
                COUNT(o.order_id) as order_count,
                COALESCE(SUM(o.total_amount), 0) as total_spent
            FROM customers c
            LEFT JOIN 'data/sample/orders.parquet' o ON c.customer_id = o.customer_id
            GROUP BY c.customer_id, c.segment, c.loyalty_points
        """).df()
        
        print(f"✓ Transformed to {len(result)} customer records")
        return result
    
    # Step 3: Define loading assets
    def asset_load_warehouse(transformed_data):
        """Load data into warehouse"""
        con.register('transformed_data', transformed_data)
        con.execute("CREATE TABLE warehouse_customer_analytics AS SELECT * FROM transformed_data")
        
        count = con.execute("SELECT COUNT(*) FROM warehouse_customer_analytics").fetchone()[0]
        print(f"✓ Loaded {count} records to warehouse")
        return count
    
    # Step 4: Create job (simulate Dagster job)
    def run_pipeline_job():
        """Execute the complete pipeline"""
        print("Running pipeline job...")
        
        # Execute assets in dependency order
        customers = asset_extract_customers()
        transformed = asset_transform_sales(customers)
        loaded = asset_load_warehouse(transformed)
        
        return loaded
    
    # Step 5: Add dependencies (implicit in function calls)
    # The dependencies are: customers → transformed → loaded
    
    # Execute the pipeline
    result = run_pipeline_job()
    
    con.close()
    return result

# Execute the solution
print("Dagster Pipeline Solution:")
print("="*50)
build_dagster_like_pipeline()